# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema available at:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

You'll reference entities (record sets, fields, columns) using their `@id` according to the Croissant schema.

In [ ]:
# Install mlcroissant (only if not installed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic dataset info
meta = dataset.metadata
print(f"Dataset: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Explore the available record sets, fields, and their `@id`s.

The `mlcroissant` library enables introspection of entity definitions, including record sets and columns. All entities are referenced by their `@id`.

In [ ]:
print("Available Record Sets (by @id):")
for record_set in dataset.metadata.record_sets.values():
    print(f"  - {record_set['@id']}: {record_set.get('name', 'Unnamed Record Set')}")

# Choose a record set to examine (use first if multiple)
record_set_ids = list(dataset.metadata.record_sets.keys())
if len(record_set_ids) > 0:
    chosen_record_set_id = record_set_ids[0]
    print(f"\nFields and columns in record set '{chosen_record_set_id}':")
    record_set = dataset.metadata.record_sets[chosen_record_set_id]
    for field in record_set['fields']:
        print(f"  * {field['@id']} (name: {field['name']}, dataType: {field.get('dataType', 'unknown')})")
        for column in field.get('columns', []):
            print(f"    - column @id: {column['@id']} (path: {column.get('path')})")
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from selected record set(s) into pandas DataFrames for analysis.

We'll use record set and field `@id`s from the above overview.

In [ ]:
# Extract all record sets (by @id)
from collections import OrderedDict
dataframes = OrderedDict()

for record_set_id in record_set_ids:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in {record_set_id}: {df.columns.tolist()}")
    print(df.head(), '\n')

# For further processing, select the main survey record set (typically first)
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

## 4. Exploratory Data Analysis (EDA)

Filter, normalize, and aggregate data on a numeric field. Use the entity `@id`s for referencing fields.

- Remove outliers (filter records)
- Normalize a numeric field (z-score)
- Group data by a categorical field

**Example:** We'll use the GAD-7 score column—its field and column `@id` can be found above; substitute accordingly.

In [ ]:
# Identify numeric and categorical field @ids
df = dataframes.get(main_record_set_id, pd.DataFrame())

# Example: Find GAD-7 score field and education level for grouping
numeric_field_id = None
group_field_id = None
for field in dataset.metadata.record_sets[main_record_set_id]['fields']:
    # Example lookups: field names containing 'gad' or 'phq' and 'education'
    fname = field['name'].lower()
    if 'gad' in fname and 'score' in fname:
        numeric_field_id = field['@id']
    if 'education' in fname:
        group_field_id = field['@id']

# Fallback if not found: use first numeric field and categorical field
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if group_field_id is None:
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            group_field_id = col
            break

# Proceed with filtering, normalization, grouping
print(f"Numeric field (GAD-7 score) @id: {numeric_field_id}")
print(f"Grouping field (education level) @id: {group_field_id}")

# Filter: GAD-7 score > 10
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:\n", filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped (mean {numeric_field_id}) by {group_field_id}:\n", grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in the dataframe.")

## 5. Visualization
Explore the data visually—for example, the distribution of GAD-7 scores and their relation to education level.
This section demonstrates field-level plotting using the relevant `@id` column names.

In [ ]:
# Basic visualizations
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load and process a FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id`. You viewed available record sets and fields, extracted data for analysis, normalized scores, and created basic visualizations of mental health survey data. For more advanced analyses, continue to use field and record set `@id`s from the Croissant schema to maintain reproducibility and clarity.